In [4]:
# ═══════════════════════════════════════════════════════════
# NOTEBOOK 03 — MODEL EVALUATION
# Marathi Mitra — Testing the fine-tuned model
#
# This notebook:
# → Loads fine-tuned model FROM Hugging Face Hub
# → Tests on SEEN words (training set)
# → Tests on UNSEEN words (generalisation test)
# → Generates before/after comparison
# → Produces output ready for README
#
# Requires: GPU (T4 minimum)
# ═══════════════════════════════════════════════════════════

print("🔍 Marathi Mitra — Model Evaluation")
print("=" * 50)
print()
print("Key question this notebook answers:")
print("→ Did the model MEMORIZE or actually LEARN?")
print()
print("If memorized: good on seen words, fails on unseen")
print("If learned:   good on BOTH seen and unseen words")

🔍 Marathi Mitra — Model Evaluation

Key question this notebook answers:
→ Did the model MEMORIZE or actually LEARN?

If memorized: good on seen words, fails on unseen
If learned:   good on BOTH seen and unseen words


In [5]:
!pip install -q "numpy>=2.0"
!pip install -q transformers==4.44.0
!pip install -q peft==0.12.0
!pip install -q bitsandbytes==0.45.3
!pip install -q accelerate==0.33.0
!pip install -q huggingface_hub

print("✅ Libraries installed!")
print("⚠️  Runtime → Restart Session → skip this cell → run Cell 2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 130.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 0.33.0 requires numpy<2.0.0,>=1.17, but you have numpy 2.4.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 1

In [ ]:
import transformers
import peft
import bitsandbytes
import accelerate
import huggingface_hub

print(f"transformers:    {transformers.__version__}")
print(f"peft:            {peft.__version__}")
print(f"bitsandbytes:    {bitsandbytes.__version__}")
print(f"accelerate:      {accelerate.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print("\n✅ All imports successful!")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN     = userdata.get("HF_TOKEN")
HF_USERNAME  = userdata.get("HF_USERNAME")
MODEL_REPO   = f"{HF_USERNAME}/marathi-mitra-phi3-v2"

login(token=HF_TOKEN)
print(f"✅ Logged in as: {HF_USERNAME}")
print(f"✅ Model repo:   {MODEL_REPO}")

In [ ]:
# ── Load directly from Hugging Face Hub ──────────────────
# This proves model works OUTSIDE of Colab training environment
# Anyone can load and use your model this way

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading fine-tuned model from Hugging Face Hub...")
print("(This is how anyone would use your model)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model.eval()

print(f"✅ Model loaded from hub!")
print(f"✅ Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)

print("Loading v2 adapter...")
model = PeftModel.from_pretrained(base, MODEL_REPO, token=HF_TOKEN)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, trust_remote_code=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ v2 Model loaded!")

In [ ]:
import re

TEST_WORDS = ["butterfly", "mother", "rain", "elephant", "school"]

GOLDEN = {
    "butterfly": {"marathi": "फुलपाखरू", "pronunciation": "Phul-pakh-roo"},
    "mother":    {"marathi": "आई",        "pronunciation": "Aa-ee"},
    "rain":      {"marathi": "पाऊस",      "pronunciation": "Paa-oos"},
    "elephant":  {"marathi": "हत्ती",     "pronunciation": "Hat-tee"},
    "school":    {"marathi": "शाळा",      "pronunciation": "Shaa-la"},
}

UNSEEN_WORDS = {
    "apple":   "सफरचंद",
    "star":    "तारा",
    "tiger":   "वाघ",
    "ocean":   "समुद्र",
    "dance":   "नृत्य",
}


def generate(word, model, tokenizer):
    prompt = f"""### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. \
When given an English word, teach it in Marathi with the word \
in Devanagari script, pronunciation, a simple story sentence, \
and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: {word}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("### Response:")[-1].strip()


def extract_fields(output):
    fields = {}
    marathi_match = re.search(r"is \*\*([^*]+)\*\*", output)
    fields["marathi_word"] = (
        marathi_match.group(1).strip() if marathi_match else ""
    )
    pronun_match = re.search(r"How to say it:\*?\*?\s*(.+)", output)
    fields["pronunciation"] = (
        pronun_match.group(1).strip() if pronun_match else ""
    )
    return fields


def evaluate_output(output, word, expected_marathi, expected_pronun):
    fields = extract_fields(output)

    # Criteria 1 — Field presence (40%)
    presence = {
        "Has Devanagari":    bool(re.search(r"[\u0900-\u097F]", output)),
        "Has pronunciation": "How to say" in output,
        "Has sentence":      "Example sentence" in output,
        "Has fun fact":      "Fun Fact" in output,
        "Has emojis":        "🌟" in output and "📢" in output,
    }
    presence_score = sum(presence.values()) / len(presence) * 100

    # Criteria 2 — Exact match (60%)
    exact = {}
    exact["Marathi word correct"] = (
        expected_marathi in fields["marathi_word"] or
        expected_marathi in output
    )
    m = fields["pronunciation"].lower().replace(" ", "").replace("-", "")
    g = expected_pronun.lower().replace(" ", "").replace("-", "")
    exact["Pronunciation correct"] = m == g
    exact_score = sum(exact.values()) / len(exact) * 100

    combined = presence_score * 0.40 + exact_score * 0.60
    return combined, presence, exact, fields


print("✅ Utilities ready")

In [ ]:
# ── Same generate and evaluate functions ─────────────────

def generate(word, model, tokenizer):
    prompt = f"""### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. \
When given an English word, teach it in Marathi with the word \
in Devanagari script, pronunciation, a simple story sentence, \
and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: {word}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("### Response:")[-1].strip()


def evaluate_output(output, expected_marathi):
    criteria = {
        "Has correct Marathi word":  expected_marathi in output,
        "Has pronunciation guide":   "How to say" in output,
        "Has example sentence":      "Example sentence" in output,
        "Has fun fact":              "Fun Fact" in output,
        "Has correct emojis":        "🌟" in output and "📢" in output,
    }
    score = sum(criteria.values()) / len(criteria) * 100
    return criteria, score


def test_word(word, expected_marathi, model, tokenizer):
    output             = generate(word, model, tokenizer)
    criteria, score    = evaluate_output(output, expected_marathi)

    print(f"\n{'─'*60}")
    print(f"📝 Word: {word.upper()}")
    print(f"\n🤖 Output:\n{output}")
    print(f"\n📊 Score: {score:.0f}%")
    for criterion, passed in criteria.items():
        print(f"   {'✅' if passed else '❌'} {criterion}")

    return output, score


print("✅ Helper functions ready")

In [ ]:
print("═" * 60)
print("TEST 1 — SEEN WORDS (in training set)")
print("Expected: HIGH scores")
print("═" * 60)

seen_outputs = {}
seen_scores  = {}
seen_total   = 0

for word, golden in GOLDEN.items():
    print(f"\n{'─'*60}")
    print(f"📝 {word.upper()}")

    output = generate(word, model, tokenizer)
    seen_outputs[word] = output
    print(f"\n🤖 Output:\n{output}")

    score, presence, exact, fields = evaluate_output(
        output, word,
        golden["marathi"],
        golden["pronunciation"],
    )
    seen_scores[word] = score
    seen_total += score

    print(f"\n📊 Criteria 1 — Field Presence (40%):")
    for k, v in presence.items():
        print(f"   {'✅' if v else '❌'} {k}")

    print(f"\n📊 Criteria 2 — Exact Match (60%):")
    print(f"   Expected: {golden['marathi']} / {golden['pronunciation']}")
    print(f"   Model:    {fields['marathi_word']} / {fields['pronunciation']}")
    for k, v in exact.items():
        print(f"   {'✅' if v else '❌'} {k}")

    print(f"\n   Score: {score:.1f}%")

seen_avg = seen_total / len(GOLDEN)
print(f"\n{'═'*60}")
print(f"SEEN WORDS AVERAGE: {seen_avg:.1f}%")
print(f"{'═'*60}")

In [ ]:
print("═" * 60)
print("TEST 2 — UNSEEN WORDS (not in training set)")
print("Tests generalisation — did model truly learn?")
print("═" * 60)

unseen_outputs = {}
unseen_scores  = {}
unseen_total   = 0

for word, expected_marathi in UNSEEN_WORDS.items():
    print(f"\n{'─'*60}")
    print(f"📝 {word.upper()}")

    output = generate(word, model, tokenizer)
    unseen_outputs[word] = output
    print(f"\n🤖 Output:\n{output}")

    # For unseen words only check field presence
    presence = {
        "Has Devanagari":    bool(re.search(r"[\u0900-\u097F]", output)),
        "Has pronunciation": "How to say" in output,
        "Has sentence":      "Example sentence" in output,
        "Has fun fact":      "Fun Fact" in output,
        "Has emojis":        "🌟" in output and "📢" in output,
    }
    presence_score = sum(presence.values()) / len(presence) * 100

    # Check if Marathi word is correct
    marathi_correct = expected_marathi in output
    exact_score     = 100 if marathi_correct else 0
    score           = presence_score * 0.40 + exact_score * 0.60

    unseen_scores[word] = score
    unseen_total += score

    print(f"\n📊 Field Presence:")
    for k, v in presence.items():
        print(f"   {'✅' if v else '❌'} {k}")

    print(f"\n📊 Marathi Word:")
    print(f"   Expected: {expected_marathi}")
    print(f"   {'✅ Correct!' if marathi_correct else '❌ Incorrect'}")
    print(f"\n   Score: {score:.1f}%")

unseen_avg = unseen_total / len(UNSEEN_WORDS)
print(f"\n{'═'*60}")
print(f"UNSEEN WORDS AVERAGE: {unseen_avg:.1f}%")
print(f"{'═'*60}")

In [ ]:
print("═" * 60)
print("FINAL EVALUATION SUMMARY — v2 MODEL")
print("═" * 60)

print(f"\n{'Word':<15} {'Type':<10} {'Score':<8} Bar")
print("─" * 50)

all_scores = []

for word, score in seen_scores.items():
    bar = "█" * int(score/10) + "░" * (10-int(score/10))
    print(f"{word:<15} {'SEEN':<10} {score:<8.1f} {bar}")
    all_scores.append(score)

print("─" * 50)

for word, score in unseen_scores.items():
    bar = "█" * int(score/10) + "░" * (10-int(score/10))
    print(f"{word:<15} {'UNSEEN':<10} {score:<8.1f} {bar}")
    all_scores.append(score)

overall = sum(all_scores) / len(all_scores)
print("═" * 50)
print(f"\n📊 RESULTS:")
print(f"   Seen words avg:   {seen_avg:.1f}%")
print(f"   Unseen words avg: {unseen_avg:.1f}%")
print(f"   Overall avg:      {overall:.1f}%")
print(f"   Generalisation gap: {seen_avg - unseen_avg:.1f}%")

print(f"\n📊 v1 vs v2 COMPARISON:")
print(f"   v1 (30 examples):  36.4%")
print(f"   v2 (250 examples): {overall:.1f}%")
improvement = overall - 36.4
print(f"   Improvement:       {improvement:+.1f}%")

if unseen_avg >= seen_avg * 0.7:
    print(f"\n✅ Good generalisation!")
    print(f"   Model learned the task, not just memorized")
else:
    print(f"\n⚠️  Some overfitting detected")
    print(f"   Seen >> Unseen — model memorized training data")
    print(f"   Solution: more diverse training data")